# Fitting Data

After collecting a dataset, comparing it with one or more theoretical models is often the first thing you'll do.
This is usually done by *fitting* the dataset to the analytic function predicted by the model you're considering.
For example, if you collect a series of current and voltage measurements, you might expect them to follow the relationship:
$$
V = IR
$$
If you fit the data with the linear function:
$$
y = mx + b
$$
then the resulting slope ($m$) is related to the resistance ($R$) of the circuit.
Assuming that $V=IR$ is a realistic model, we would expect the y-intercept ($b$) to be zero here.
Generally, by choosing a realistic analytic function, the fitted parameters will be related to important physical properties of the system.

We will use the `curve_fit()` function provided by the SciPy library.
This function uses the non-linear least squares method to fit a function to data.
To install this, open a terminal within VS Code, wait for the virtual enivornement to activate, then enter:
```bash
python -m pip install scipy
```

## Least Squares Method

The Wikipedia article [Least Squares](https://en.wikipedia.org/wiki/Least_squares) provides more details and additional references for the method; a reduced summary is given below.

The goal of this method is to adjust the free parameters of a function to best fit data.
The inputs are:
- a data set consisting of $n$ points, $(x_i, y_i)$ for $i = 1, ..., n$, and
- a function of the form $f(x; \mathbf{\beta})$, where $\mathbf{\beta}$ is a vector of $m$ adjustable parameters.

For a given set of parameters, the residual of a given data point $(x_i, y_i)$ with respect to the model function is:
$$
r_i = y_i - f(x; \mathbf{\beta})
$$
The least squares method then finds the optimal parameters by minimizing the sum:
$$
S = \sum_{i=1}^{n} r_i^2
$$
In other words, the method computes $S$ for many choices of parameters ($\mathbf{\beta}$) and defines the optimal parameters as those that result in the smallest value of $S$.
While a closed form solution exists for function linear in the free parameters ($\mathbf{\beta}$), there is generally no closed-form solution for the non-linear case.
Efficiently sampling the parameter space in order to determine the optimal parameters is a rich and interesting topic which is covered in PHYS-P 410.

The first figure below shows data (black points) scattered around a model function (red line), with the residuals indicated by thinner red lines.
A second figure shows only the residuals, which appear to randomly fluctuate around the model function, indicating it provides a good fit of the data.

<p align="center">
  <img src="./Linear_regression_residuals.png" alt="Example of data scattered around a linear function" title="Linear Regression Residuals">
</p>


## Ohm's Law Example

The file `ohms_data.csv` contains data generated assuming the relationship $V=IR$ with an unknown resistance, with voltage–current pairs separated by a comma on each line.
(The file extension `csv` stands for comma-separated values.)
Let's start by exploiting the `np.genfromtxt()` function to easily read the data, specifying that the data is separated by commas and that we want to skip the first line which contains header information.

In [ ]:
import numpy as np

# Use NumPy function to read CSV data from file
data = np.genfromtxt('ohms_data.csv', delimiter=',',skip_header=1)

print(data)

voltages = data[:,0] # slicing out row 0 provides voltage data
currents = data[:,2] # slicing out row 2 provides current data


Since it's hard to parse raw data dumps like this, let's quickly make a plot to get an idea what the data looks like.

In [ ]:
import matplotlib.pyplot as plt

# Plot voltage vs current, since the slope is expected to be R
fig, ax = plt.subplots()
ax.scatter(currents, voltages, c='black')
ax.set_xlabel('Current [A]')
ax.set_ylabel('Voltage [V]')

This looks fairly linear, so let's try fitting a line of the form $y = mx+b$ to it.
First, we will need to define the linear function we want to fit:

In [ ]:
def linear(x, m, b):
    """Simple linear function with slope and y-intercept parameters"""
    return m*x + b

The independent variable ($x$) is the first parameter, and any free parameters are provided after this.
The `curve_fit()` function will check how many free parameters there are and try to find optimal values for all of them.

Now we can pass the data and the fit function to the `curve_fit()` function included in SciPy's `optimize` module:

In [ ]:
from scipy.optimize import curve_fit

# Fit data to linear function using least squares regression to extract slope -> R
popt, pcov = curve_fit(linear, currents, voltages)

print(f'The fit indicates a resistance of {popt[0]} and a y-offset (sensor error) of {popt[1]}')

We started by importing the `curve_fit()` function from `scipy.optimize`.
Next, we passed it the function we're fitting to the data.
Finally, we passed the array of independent variable points ($I_i$) and the dependent variable points ($V_i$)

This function returns two variables: `popt` and `pcov`.
The `popt` variables contains an array of the optimal parameters and `pcov` contains a covariance matrix with correlation information.

Since we expect the data follow the model $V=IR$, the first parameter ($m$) corresponds to the optimal resistance ($R$) from the fit.
It's about $10.2\Omega$, which is good, considering I generated the data assuming a resistance of $10\Omega$!

To check how the fit looks compared to the data, we can make an array of expected points for the same current values we fit:

In [ ]:
# Use vectorization to make array of fit voltages
fit_voltages = linear(currents, popt[0], popt[1])

# Plot both 
fig, ax = plt.subplots()
ax.scatter(currents, voltages, c='black', label='Measured')
ax.plot(currents, fit_voltages, c='red', label='Fit')
ax.set_xlabel('Current [A]')
ax.set_ylabel('Voltage [V]')
ax.legend()


## Including Uncertainties

As your freshman lab instructor might have emphasized, you should always provide error bars on the data points you collect!
In the file we've been looking at, these are stored in the second column, which we previously ignored.

To include them in the fit, we can pass them to the `sigma` parameter of `curve_fit()`.
This updates the sum that is minimized in order to find the optimal parameters to be:
$$
\chi^2 = \sum_{i=1}^{n} \left( \frac{r_i}{\sigma_i} \right)^2
$$
Essentially, the residual ($r_i$) is now considered relative to the measured uncertainty in each bin.
A large residual on a measurement with large uncertainty carries the same weight as a small residual on a measurement with a small uncertainty.

Let's update our code above to include the uncertainties in the plot as well as the fit:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Read data from file and store in NumPy arrays
data = np.genfromtxt('ohms_data.csv', delimiter=',',skip_header=1)

voltages = data[:,0] # slicing out row 0 provides voltage data
voltage_errors = data[:,1] # slicing out row 1 provides uncertainty on voltage data
currents = data[:,2] # slicing out row 2 provides current data

# Fit linear model to data, including uncertainty on measured voltage
def linear(x, m, b):
    """Simple linear function with slope and y-intercept parameters"""
    return m*x + b

(m,b), pcov = curve_fit(linear, currents, voltages, sigma=voltage_errors, absolute_sigma=True)

# Make a plot of the results
fit_voltages = linear(currents, m, b)

fig, ax = plt.subplots()
ax.errorbar(currents, voltages, yerr=voltage_errors, c='black', fmt='o', label='Measured')
ax.plot(currents, fit_voltages, c='red', label='Fit')
ax.set_xlabel('Current [A]')
ax.set_ylabel('Voltage [V]')
ax.legend()

# The slope of the fit is the resistance
print(f'The fit indicates a resistance of {m} and a y-offset (sensor error) of {b}')

## Initial Parameters

By default, `curve_fit()` assumes that the initial value of all free parameters is 1.
It then explores values around this, and moves in the direction that provides a smaller $S$ or $\chi^2$.
Sometimes, if this initial value is too far away from the optimal parameter value, the fit will fail to converge.

To get around this, you can supply a better initial guess using the `p0` parameter.
This doesn't need to be precise, and how you estimate it can vary depending on what you're fitting.
In the example above, we could make a rough estimate of the slope by using the first and last current.

This would look something like:

In [ ]:
# Use numpy functions to find index of max and min current values
min_index = np.argmin(currents)
max_index = np.argmax(currents)

# Calculate slope using these values; assume y-intercept is 0
m0 = (voltages[max_index] - voltages[min_index]) / (currents[max_index] - currents[min_index])
b0 = 0

# Set initial guess for parameters and pass to curve_fit
p0 = [m0, b0]
(m,b), pcov = curve_fit(linear, currents, voltages, sigma=voltage_errors, absolute_sigma=True, p0=p0)

## Practice

In a gas, molecule speeds follow the Maxwell-Boltzmann distribution, but at high speeds or in specific velocity components (like just the 
$x$-direction), the data looks perfectly Gaussian.

An experiment was run that counted the number of particles moving in the $x$-direction at different speed ranges. For example, it counted how many particles had a speed from approximately 269 m/s to 307 m/s, and stored that number in a CSV file paired with the average speed (288 m/s in this case). The full results are stored in the file `gas_data.csv`.

Make a plot of this data and fit a Gaussian distribution to it. Judge (by eye) if the Gaussian distribution is a good fit, and print the optimal mean the fit reports. Remember that the Gaussian distribution has the form:
$$
f(x) = A e^{-(x-\mu)^2 / 2\sigma^2}
$$